# README

- **Author**: Codex
- **Created At**: 2026-10-13
- **Last Modified At**: 2026-10-13

## What does this file do?

先整理訂單資料，再把商品名稱和日期資訊合併進來。

原始訂單有 122 列，其中 2 列完全重複、3 筆購買件數缺漏、2 筆單價為負。這些都是假資料裡刻意加入的問題，這次直接刪除，最後留下 115 筆。

清理後，去掉商品類別前後的空白，將件數改成整數，並用「件數 × 單價」算出訂單金額。接著按照 `category` 加入商品名稱，再按照 `order_date` 加入星期及平日／週末資訊。

## What does this file take?

- [訂單資料](../raw/demo_orders.csv)：122 列，包含日期、商品類別、件數和單價。[資料說明](../raw/demo_orders_documentation.ipynb)
- [商品類別表](../raw/demo_categories.csv)：3 個類別，對應飲品、點心和輕食。[資料說明](../raw/demo_categories_documentation.ipynb)
- [日期表](../raw/demo_calendar.csv)：2026 年 9 月的 30 天，記錄星期及平日／週末。[資料說明](../raw/demo_calendar_documentation.ipynb)

## What does this file output?

結果存到 `data/processed/demo_orders_clean.csv`，共 115 筆、9 欄。每列仍是一筆訂單，合併後沒有多出或漏掉訂單。

欄位定義放在 [整理後資料說明](../processed/demo_orders_documentation.ipynb)。後面的 EDA 直接讀取這份 CSV；重跑本文件時會重新產生整理後資料，三份原始檔不會被修改。


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# 找到專案資料夾，從不同位置開啟 notebook 都能讀到資料。
project_root = Path.cwd().resolve()
while not ((project_root / "AGENTS.md").is_file() and (project_root / "templates").is_dir()):
    if project_root == project_root.parent:
        raise FileNotFoundError("請從專案資料夾內開啟 notebook。")
    project_root = project_root.parent


In [2]:
raw = pd.read_csv(project_root / "data/raw/demo_orders.csv")
categories = pd.read_csv(project_root / "data/raw/demo_categories.csv")
calendar = pd.read_csv(project_root / "data/raw/demo_calendar.csv")

print(f"訂單：{len(raw)} 列")
print(f"商品類別：{len(categories)} 列")
print(f"日期：{len(calendar)} 列")


訂單：122 列
商品類別：3 列
日期：30 列


In [3]:
clean = raw.drop_duplicates().copy()
duplicates = len(raw) - len(clean)

# 件數缺漏或單價不合理的訂單，這次先不納入。
missing_quantity = clean["quantity"].isna()
invalid_price = clean["unit_price_twd"] <= 0
clean = clean.loc[~missing_quantity & ~invalid_price].copy()

clean["category"] = clean["category"].str.strip()
clean["order_date"] = pd.to_datetime(clean["order_date"]).dt.strftime("%Y-%m-%d")
assert (clean["quantity"] % 1 == 0).all(), "件數有小數，請先確認。"
clean["quantity"] = clean["quantity"].astype(int)
clean["order_amount_twd"] = clean["quantity"] * clean["unit_price_twd"]
clean = clean.sort_values("order_id").reset_index(drop=True)
rows_before_merge = len(clean)

# 對照表的每個類別、日期只能出現一次，否則合併會增加訂單列數。
clean = clean.merge(categories, on="category", how="left", validate="many_to_one")
assert clean["category_name"].notna().all(), "有商品類別沒有配對到。"
clean = clean.merge(calendar, on="order_date", how="left", validate="many_to_one")
assert clean[["weekday", "day_type"]].notna().all().all(), "有日期沒有配對到。"
assert len(clean) == rows_before_merge
assert clean["order_id"].is_unique
assert not clean.isna().any().any()

print(f"刪除重複：{duplicates} 列")
print(f"排除件數缺漏：{missing_quantity.sum()} 筆")
print(f"排除非正單價：{invalid_price.sum()} 筆")
print(f"合併完成：{len(clean)} 筆、{len(clean.columns)} 欄")


刪除重複：2 列
排除件數缺漏：3 筆
排除非正單價：2 筆
合併完成：115 筆、9 欄


In [4]:
output_path = project_root / "data/processed/demo_orders_clean.csv"
clean.to_csv(output_path, index=False)

# 再讀一次，確認存出的資料和整理結果相同。
saved = pd.read_csv(output_path)
pd.testing.assert_frame_equal(clean, saved)
print(f"已存到 data/processed/{output_path.name}")
display(saved.head(10))


已存到 data/processed/demo_orders_clean.csv


,order_id,order_date,category,quantity,unit_price_twd,order_amount_twd,category_name,weekday,day_type
0,D0001,2026-09-01,A,1,80,80,飲品,星期二,平日
1,D0002,2026-09-02,B,3,97,291,點心,星期三,平日
2,D0003,2026-09-03,C,5,114,570,輕食,星期四,平日
3,D0004,2026-09-04,A,2,131,262,飲品,星期五,平日
4,D0005,2026-09-05,B,4,148,592,點心,星期六,週末
5,D0006,2026-09-06,C,1,165,165,輕食,星期日,週末
6,D0007,2026-09-07,A,3,182,546,飲品,星期一,平日
7,D0008,2026-09-08,B,5,199,995,點心,星期二,平日
8,D0010,2026-09-10,A,4,233,932,飲品,星期四,平日
9,D0011,2026-09-11,B,1,250,250,點心,星期五,平日
